In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from supabase import create_client, Client

In [2]:
load_dotenv()
url = os.environ.get("SUPABASE_URL").strip()
key = os.environ.get("SUPABASE_KEY").strip()

In [6]:
supabase: Client = create_client(url, key)
response = supabase.table('matches').select('*').limit(10000).execute()
df_models = pd.DataFrame(response.data)

print(f"success: downloaded {len(df_models)} records")

success: downloaded 8445 records


In [9]:
df_models = df_models.sort_values('date').reset_index(drop=True)

In [10]:
features = [
    'b365h', 'b365d', 'b365a',
    'home_team_goals_avg_last_5', 'away_goals_avg_last_5', 
    'home_goals_conceded_avg_last_5', 'away_goals_conceded_avg_last_5',
    'ht_home_goals_avg_last_5', 'ht_away_goals_avg_last_5',
    'home_form_1h_last_5', 'away_form_1h_last_5', 'home_form_2h_last_5', 'away_form_2h_last_5',
    'home_shots_avg_last_5', 'away_shots_avg_last_5', 
    'home_shots_target_avg_last_5', 'away_shots_target_avg_last_5',
    'home_red_cards_avg_last_5', 'away_red_cards_avg_last_5',
    'home_points_avg_last_5', 'away_points_avg_last_5', 
    'home_overall_points_last_5', 'away_overall_points_last_5'
]



In [12]:
missing_values = df_models.isnull().mean() * 100
missing_values[missing_values > 0]

home_team_goals_avg_last_5        1.539372
away_goals_avg_last_5             1.539372
ht_home_goals_avg_last_5          1.539372
ht_away_goals_avg_last_5          1.539372
home_form_1h_last_5               1.539372
away_form_1h_last_5               1.539372
home_form_2h_last_5               1.539372
away_form_2h_last_5               1.539372
home_shots_avg_last_5             1.539372
away_shots_avg_last_5             1.539372
home_shots_target_avg_last_5      1.539372
away_shots_target_avg_last_5      1.539372
home_red_cards_avg_last_5         1.539372
away_red_cards_avg_last_5         1.539372
home_goals_conceded_avg_last_5    1.539372
away_goals_conceded_avg_last_5    1.539372
home_points_avg_last_5            1.539372
away_points_avg_last_5            1.539372
home_overall_points_last_5        0.746004
away_overall_points_last_5        0.793369
dtype: float64

In [13]:
df_clean_for_ml = df_models.dropna(subset=features + ['target'])

In [14]:
X = df_clean_for_ml[features]
y = df_clean_for_ml['target']
split_index = int(len(X) * 0.8)
X_train = X.iloc[:split_index]
y_train = y.iloc[:split_index]
X_test = X.iloc[split_index:]
y_test = y.iloc[split_index:]
